# Basira AI — 01: Exploratory Data Analysis (EDA)

**Diabetic Retinopathy Screening in Low-Resource Settings**

This notebook measures real dataset characteristics across all 4 benchmark sources:
1. **APTOS 2019** (Indian screening programme)
2. **EyePACS** (US screening programme — resized 15 mirror)
3. **Messidor-2** (French hospital data — external test only)
4. **IDRiD** (Indian eye hospital — external test only)

> **Project Rule:** Never invent numbers. All statistics, image sizes, grade distributions,
> and referable DR prevalences in `reports/eda_summary.json` are computed directly by code.

## 1. Environment Setup & Configuration

In [ ]:
import json
import pathlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Mount Drive if running on Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Ensure src/ is on path
REPO_ROOT = pathlib.Path.cwd()
if (REPO_ROOT / 'src').exists() and str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from basira.config import load_config, print_config
from basira.data.loaders import load_aptos, load_eyepacs, load_idrid, load_messidor2
from basira.data.preprocess import crop_and_enhance

cfg = load_config()
print_config(cfg)

## 2. Load Unified Metadata across Datasets

Every loader returns standardized columns: `path`, `image_id`, `grade`, `referable`, `patient_id`, `eye`, `source`.

In [ ]:
print('Loading dataset metadata from config paths...')
df_aptos = load_aptos(cfg)
df_ep_train = load_eyepacs(cfg, split='train')
df_ep_test = load_eyepacs(cfg, split='test')
df_mess = load_messidor2(cfg)
df_idrid = load_idrid(cfg)

datasets = {
    'APTOS 2019': df_aptos,
    'EyePACS (train)': df_ep_train,
    'EyePACS (test)': df_ep_test,
    'Messidor-2': df_mess,
    'IDRiD': df_idrid,
}

for name, df in datasets.items():
    print(f'  {name:<18} : {len(df):>6} rows | columns: {list(df.columns)}')

## 3. Grade Distribution & Referable DR Prevalence

Primary screening task: **Referable DR** (Grade $\ge$ 2: Moderate, Severe, Proliferative DR).
Secondary task: **5-class DR Grading** (0 = No DR, 1 = Mild, 2 = Moderate, 3 = Severe, 4 = Proliferative).

In [ ]:
summary_stats = {}

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharey=False)
grade_labels = ['0: None', '1: Mild', '2: Mod', '3: Sev', '4: PDR']

for ax_idx, (name, df) in enumerate(datasets.items()):
    counts = df['grade'].value_counts().reindex(range(5), fill_value=0)
    total = len(df)
    ref_count = df['referable'].sum()
    ref_prev = ref_count / total if total > 0 else 0.0

    summary_stats[name] = {
        'total_images': int(total),
        'grade_counts': {int(k): int(v) for k, v in counts.items()},
        'grade_proportions': {int(k): round(float(v / total), 4) for k, v in counts.items()},
        'referable_count': int(ref_count),
        'referable_prevalence': round(float(ref_prev), 4),
    }

    # Plot bar chart
    colors = ['#2b5c8f', '#2b5c8f', '#d95f02', '#d95f02', '#d95f02']  # Orange for referable
    axes[ax_idx].bar(range(5), counts.values, color=colors, edgecolor='black', alpha=0.85)
    axes[ax_idx].set_title(f'{name}\n(Ref: {ref_prev*100:.1f}%)', fontsize=10, fontweight='bold')
    axes[ax_idx].set_xticks(range(5))
    axes[ax_idx].set_xticklabels(grade_labels, rotation=45, ha='right', fontsize=8)
    axes[ax_idx].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Print clean summary table
print('=' * 85)
print(f"{'Dataset':<18} | {'Total':>7} | {'Grade 0':>7} | {'Grade 1':>7} | {'Grade 2':>7} | {'Grade 3':>7} | {'Grade 4':>7} | {'Ref. %':>7}")
print('=' * 85)
for name, st in summary_stats.items():
    gc = st['grade_counts']
    print(f"{name:<18} | {st['total_images']:>7} | {gc[0]:>7} | {gc[1]:>7} | {gc[2]:>7} | {gc[3]:>7} | {gc[4]:>7} | {st['referable_prevalence']*100:>6.1f}%")
print('=' * 85)

## 4. Measured Image Dimension Statistics

> **Rule:** Do not assume image dimensions; measure them directly.
> Note: EyePACS dimensions below reflect the **resized-15 mirror** dataset (approx. 512–1024px), not the raw ~35GB original files.

In [ ]:
SAMPLE_PER_DATASET = 500
print(f'Sampling up to {SAMPLE_PER_DATASET} images per dataset to measure actual dimensions...')

size_summary = {}

for name, df in datasets.items():
    sample_df = df.sample(n=min(SAMPLE_PER_DATASET, len(df)), random_state=42)
    widths, heights = [], []

    for _, row in sample_df.iterrows():
        try:
            with Image.open(row['path']) as im:
                w, h = im.size
                widths.append(w)
                heights.append(h)
        except Exception:
            continue

    widths = np.array(widths)
    heights = np.array(heights)

    stats_w = {
        'min': int(widths.min()),
        'max': int(widths.max()),
        'mean': round(float(widths.mean()), 1),
        'median': int(np.median(widths)),
        'std': round(float(widths.std()), 1),
    }
    stats_h = {
        'min': int(heights.min()),
        'max': int(heights.max()),
        'mean': round(float(heights.mean()), 1),
        'median': int(np.median(heights)),
        'std': round(float(heights.std()), 1),
    }

    size_summary[name] = {
        'sampled_count': len(widths),
        'width_stats': stats_w,
        'height_stats': stats_h,
        'note': 'Resized-15 mirror dataset' if 'EyePACS' in name else 'Original camera resolution',
    }

    print(f'  {name:<18} : Sampled {len(widths):>3} imgs | Width: {stats_w["mean"]}px (min {stats_w["min"]}, max {stats_w["max"]}) | Height: {stats_h["mean"]}px (min {stats_h["min"]}, max {stats_h["max"]})')

print('\nMeasured image sizes confirmed.')

## 5. EyePACS Patient-Level Multi-Eye Analysis

Check the distribution of images per patient in EyePACS. Verify that **no patient has more than 2 images** (left and right eye).

In [ ]:
print('Analyzing EyePACS patient image frequencies...')
ep_patient_counts = df_ep_train.groupby('patient_id').size()

total_patients = len(ep_patient_counts)
patients_with_1_eye = int((ep_patient_counts == 1).sum())
patients_with_2_eyes = int((ep_patient_counts == 2).sum())
patients_with_gt2 = int((ep_patient_counts > 2).sum())

print(f'  Total unique patients: {total_patients}')
print(f'  Patients with 1 image (monocular) : {patients_with_1_eye:>5} ({patients_with_1_eye/total_patients*100:.1f}%)')
print(f'  Patients with 2 images (binocular) : {patients_with_2_eyes:>5} ({patients_with_2_eyes/total_patients*100:.1f}%)')
print(f'  Patients with >2 images            : {patients_with_gt2:>5}')

assert patients_with_gt2 == 0, f'VIOLATION: Found {patients_with_gt2} patients with > 2 images in EyePACS!'
print('✓ Verified: No EyePACS patient has more than 2 images.')

eyepacs_patient_stats = {
    'total_unique_patients': total_patients,
    'patients_with_1_eye': patients_with_1_eye,
    'patients_with_2_eyes': patients_with_2_eyes,
    'patients_with_gt2': patients_with_gt2,
}

## 6. APTOS Perceptual Hash Deduplication Results

Read the deduplication summary produced by `basira.data.dedup` (instead of re-running the 256-bit hash search).

In [ ]:
dedup_summary_path = pathlib.Path(cfg.output.reports) / 'aptos_dedup_summary.json'
local_dedup_path = REPO_ROOT / 'reports' / 'aptos_dedup_summary.json'

dedup_info = None
for p in [dedup_summary_path, local_dedup_path]:
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            dedup_info = json.load(f)
        print(f'Loaded deduplication results from: {p}')
        break

if dedup_info is not None:
    print(f"  Original APTOS images   : {dedup_info['original_count']}")
    print(f"  Deduplicated images     : {dedup_info['deduped_count']}")
    print(f"  Dropped (Agreement)     : {dedup_info['agreement_dropped']}")
    print(f"  Dropped (Conflict)      : {dedup_info['conflict_dropped']}")
    print(f"  Total Dropped           : {dedup_info['total_dropped']}")
    print(f"  Active Threshold (d)    : {dedup_info.get('distance_threshold', 'N/A')}")
    print(f"  Hamming NN Distribution : {dedup_info['nn_distance_distribution']}")
    if 'counts_by_distance' in dedup_info:
        print('  Pair Counts by Distance (0 to 8):')
        for dist, cnt in dedup_info['counts_by_distance'].items():
            print(f'    d = {dist:>2} : {cnt:>4} pairs')
else:
    print('ℹ️ Deduplication summary not run yet. Run scripts/create_splits.py to generate.')

## 7. Sample Image Grid & Preprocessing Demo

Visualizing sample images across DR grades 0 to 4 and demonstrating Stage 1 preprocessing (black border crop + square pad + CLAHE in LAB space).

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(12, 14))

for g in range(5):
    grade_df = df_aptos[df_aptos['grade'] == g]
    sample_rows = grade_df.sample(min(2, len(grade_df)), random_state=42).reset_index(drop=True)

    for col_idx in range(2):
        if col_idx < len(sample_rows):
            row = sample_rows.iloc[col_idx]
            im_raw = Image.open(row['path']).convert('RGB')
            im_prep = crop_and_enhance(im_raw, target_size=256, apply_clahe=True)

            # Raw
            axes[g, col_idx*2].imshow(im_raw)
            axes[g, col_idx*2].set_title(f'Grade {g} (Raw)\n{row["image_id"]}', fontsize=8)
            axes[g, col_idx*2].axis('off')

            # Preprocessed
            axes[g, col_idx*2 + 1].imshow(im_prep)
            axes[g, col_idx*2 + 1].set_title(f'Grade {g} (Crop+CLAHE)\n256x256', fontsize=8, color='navy')
            axes[g, col_idx*2 + 1].axis('off')

plt.suptitle('APTOS Samples per DR Grade: Raw vs. Preprocessed (Crop + Square Pad + LAB CLAHE)', fontsize=12, y=1.00)
plt.tight_layout()
plt.show()

### 7.1 EyePACS Train & Test Preprocessing Demo

Showing 3 random samples each from EyePACS train and EyePACS test (Raw vs. Preprocessed, fixed seed=42),
plus an extra comparison row demonstrating that train and test images from the mirror (which often have
different raw resolutions) are mapped to consistent 256x256 square inputs.

In [ ]:
fig, axes = plt.subplots(7, 2, figsize=(8, 20))

# Rows 0-2: EyePACS Train (3 samples, fixed seed=42)
sample_ep_train = df_ep_train.sample(3, random_state=42).reset_index(drop=True)
for i in range(3):
    row = sample_ep_train.iloc[i]
    im_raw = Image.open(row['path']).convert('RGB')
    im_prep = crop_and_enhance(im_raw, target_size=256, apply_clahe=True)

    axes[i, 0].imshow(im_raw)
    axes[i, 0].set_title(f"EyePACS Train Raw ({im_raw.size[0]}x{im_raw.size[1]})\n{row['image_id']} (Grade {row['grade']})", fontsize=8)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(im_prep)
    axes[i, 1].set_title("EyePACS Train Preprocessed (256x256)\nCrop + Pad + CLAHE", fontsize=8, color='navy')
    axes[i, 1].axis('off')

# Rows 3-5: EyePACS Test (3 samples, fixed seed=42)
sample_ep_test = df_ep_test.sample(3, random_state=42).reset_index(drop=True)
for i in range(3):
    row = sample_ep_test.iloc[i]
    im_raw = Image.open(row['path']).convert('RGB')
    im_prep = crop_and_enhance(im_raw, target_size=256, apply_clahe=True)

    axes[i + 3, 0].imshow(im_raw)
    axes[i + 3, 0].set_title(f"EyePACS Test Raw ({im_raw.size[0]}x{im_raw.size[1]})\n{row['image_id']} (Grade {row['grade']})", fontsize=8)
    axes[i + 3, 0].axis('off')

    axes[i + 3, 1].imshow(im_prep)
    axes[i + 3, 1].set_title("EyePACS Test Preprocessed (256x256)\nCrop + Pad + CLAHE", fontsize=8, color='navy')
    axes[i + 3, 1].axis('off')

# Row 6: Extra row comparing Train image vs. Test image after preprocessing
row_comp_tr = df_ep_train.sample(1, random_state=99).iloc[0]
row_comp_te = df_ep_test.sample(1, random_state=99).iloc[0]

im_tr_raw = Image.open(row_comp_tr['path']).convert('RGB')
im_te_raw = Image.open(row_comp_te['path']).convert('RGB')
im_tr_prep = crop_and_enhance(im_tr_raw, target_size=256, apply_clahe=True)
im_te_prep = crop_and_enhance(im_te_raw, target_size=256, apply_clahe=True)

axes[6, 0].imshow(im_tr_prep)
axes[6, 0].set_title(f"[EXTRA ROW] Train Post-Prep (256x256)\nRaw size: {im_tr_raw.size[0]}x{im_tr_raw.size[1]} | Grade {row_comp_tr['grade']}", fontsize=8, color='darkgreen', fontweight='bold')
axes[6, 0].axis('off')

axes[6, 1].imshow(im_te_prep)
axes[6, 1].set_title(f"[EXTRA ROW] Test Post-Prep (256x256)\nRaw size: {im_te_raw.size[0]}x{im_te_raw.size[1]} | Grade {row_comp_te['grade']}", fontsize=8, color='darkgreen', fontweight='bold')
axes[6, 1].axis('off')

plt.suptitle("EyePACS: Train vs. Test Preprocessing & Mirror Size Normalization", fontsize=11, y=1.00)
plt.tight_layout()
plt.show()

### 7.2 External Test Sets Preprocessing Demo: Messidor-2 & IDRiD

Showing 3 random samples each from Messidor-2 and IDRiD (Raw vs. Preprocessed, fixed seed=42).

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(8, 17))

# Rows 0-2: Messidor-2 (3 samples, fixed seed=42)
sample_mess = df_mess.sample(3, random_state=42).reset_index(drop=True)
for i in range(3):
    row = sample_mess.iloc[i]
    im_raw = Image.open(row['path']).convert('RGB')
    im_prep = crop_and_enhance(im_raw, target_size=256, apply_clahe=True)

    axes[i, 0].imshow(im_raw)
    axes[i, 0].set_title(f"Messidor-2 Raw ({im_raw.size[0]}x{im_raw.size[1]})\n{row['image_id']} (Grade {row['grade']})", fontsize=8)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(im_prep)
    axes[i, 1].set_title("Messidor-2 Preprocessed (256x256)\nCrop + Pad + CLAHE", fontsize=8, color='navy')
    axes[i, 1].axis('off')

# Rows 3-5: IDRiD (3 samples, fixed seed=42)
sample_idrid = df_idrid.sample(3, random_state=42).reset_index(drop=True)
for i in range(3):
    row = sample_idrid.iloc[i]
    im_raw = Image.open(row['path']).convert('RGB')
    im_prep = crop_and_enhance(im_raw, target_size=256, apply_clahe=True)

    axes[i + 3, 0].imshow(im_raw)
    axes[i + 3, 0].set_title(f"IDRiD Raw ({im_raw.size[0]}x{im_raw.size[1]})\n{row['image_id']} (Grade {row['grade']})", fontsize=8)
    axes[i + 3, 0].axis('off')

    axes[i + 3, 1].imshow(im_prep)
    axes[i + 3, 1].set_title("IDRiD Preprocessed (256x256)\nCrop + Pad + CLAHE", fontsize=8, color='navy')
    axes[i + 3, 1].axis('off')

plt.suptitle("External Test Sets: Messidor-2 & IDRiD (Raw vs. Preprocessed)", fontsize=11, y=1.00)
plt.tight_layout()
plt.show()

## 8. Export Verified EDA Metrics to `reports/eda_summary.json`

In [ ]:
final_eda_report = {
    'datasets': summary_stats,
    'measured_image_dimensions': size_summary,
    'eyepacs_patient_statistics': eyepacs_patient_stats,
    'aptos_deduplication': dedup_info if dedup_info is not None else 'not run yet',
}

# Save to local reports/ and Drive
local_report_path = REPO_ROOT / 'reports' / 'eda_summary.json'
local_report_path.parent.mkdir(parents=True, exist_ok=True)
with open(local_report_path, 'w', encoding='utf-8') as f:
    json.dump(final_eda_report, f, indent=2)

drive_report_path = pathlib.Path(cfg.output.reports) / 'eda_summary.json'
try:
    drive_report_path.parent.mkdir(parents=True, exist_ok=True)
    with open(drive_report_path, 'w', encoding='utf-8') as f:
        json.dump(final_eda_report, f, indent=2)
    print(f'✓ Exported to Google Drive: {drive_report_path}')
except Exception as e:
    print(f'Drive export skipped: {e}')

print(f'✓ Exported to local reports: {local_report_path}')
print('Phase 2 EDA complete.')